# 13 — Learning Memory & Personalized Recommendations

This notebook demonstrates the **RecommendationEngine** — a personalized study
advisor that combines progress tracking, knowledge graph structure, and spaced
repetition scheduling to generate actionable study plans.

## Features
- **Weak Topic Detection**: Identifies topics needing practice based on scores
- **Prerequisite-Aware Suggestions**: Recommends next topics when prerequisites are met
- **Spaced Repetition Integration**: Surfaces flashcards due for review
- **Daily Study Plans**: Generates structured plans with time estimates
- **Revision Timing**: Flags topics that haven't been reviewed recently

In [ ]:
import sys
sys.path.insert(0, "..")

from src.memory import ProgressTracker, RecommendationEngine, SpacedRepetitionScheduler
from src.store import KnowledgeGraph
from models import Concept, ConceptRelationship, Difficulty

## Setup

Initialize the components: progress tracker, knowledge graph, spaced repetition scheduler,
and the recommendation engine that ties them together.

In [ ]:
# Initialize components
tracker = ProgressTracker(state_file="./demo_progress.json")
tracker.reset()  # Start fresh for the demo

scheduler = SpacedRepetitionScheduler(state_file="./demo_sr.json")
scheduler.reset()

# Build a small knowledge graph
kg = KnowledgeGraph()
concepts = [
    Concept(id="python_basics", name="Python Basics", definition="Variables, loops, functions", difficulty=Difficulty.EASY, topics=["programming"]),
    Concept(id="data_structures", name="Data Structures", definition="Lists, dicts, sets", difficulty=Difficulty.MEDIUM, topics=["programming"]),
    Concept(id="algorithms", name="Algorithms", definition="Sorting, searching", difficulty=Difficulty.MEDIUM, topics=["programming"]),
    Concept(id="machine_learning", name="Machine Learning", definition="ML fundamentals", difficulty=Difficulty.HARD, topics=["ai"]),
]
relationships = [
    ConceptRelationship(source_concept="python_basics", target_concept="data_structures", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="python_basics", target_concept="algorithms", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="data_structures", target_concept="machine_learning", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="algorithms", target_concept="machine_learning", relationship_type="prerequisite"),
]
kg.add_concepts(concepts)
kg.add_relationships(relationships)

# Create the recommendation engine
engine = RecommendationEngine(
    progress_tracker=tracker,
    knowledge_graph=kg,
    scheduler=scheduler,
)
print("Components initialized!")

## Personalized Recommendations

### Beginner State
With no progress recorded, the engine suggests starting with root topics (those with no prerequisites).

In [ ]:
# Beginner: no progress yet
recs = engine.get_study_recommendations()
print("=== Beginner Recommendations ===")
for r in recs:
    print(f"  [{r['priority']}] {r['action'].upper()}: {r['topic']} — {r['reason']}")

### After Some Progress
Once we master Python Basics, new topics become available.

In [ ]:
# Record mastery of Python Basics
tracker.record_score("Python Basics", 92)
tracker.record_score("Python Basics", 88)

# Record a weak score in Data Structures
tracker.record_score("Data Structures", 45)

# Add some flashcards due for review
scheduler.add_card("python_variables")
scheduler.add_card("list_comprehensions")

recs = engine.get_study_recommendations()
print("=== After Some Progress ===")
for r in recs:
    print(f"  [{r['priority']}] {r['action'].upper()}: {r['topic']} — {r['reason']}")

### Next Topic Suggestions
Topics whose prerequisites are all mastered or familiar.

In [ ]:
next_topics = engine.suggest_next_topics()
print("Topics ready to learn:")
for topic in next_topics:
    print(f"  • {topic}")

### Revision Suggestions
Topics that haven't been reviewed recently and may need a refresher.

In [ ]:
revision = engine.suggest_revision_topics()
print("Topics needing revision:")
if revision:
    for topic in revision:
        print(f"  • {topic}")
else:
    print("  All topics are up to date!")

### Daily Study Plan
A structured plan combining all signals into an actionable schedule.

In [ ]:
plan = engine.get_daily_plan()
print("=== Daily Study Plan ===")
print(f"\nFlashcards to review ({len(plan['review_cards'])})")
for card in plan['review_cards']:
    print(f"  📋 {card}")

print(f"\nWeak topics to practice ({len(plan['weak_topics'])})")
for topic in plan['weak_topics']:
    print(f"  🔴 {topic}")

print(f"\nNew topics to explore ({len(plan['next_topics'])})")
for topic in plan['next_topics']:
    print(f"  🟢 {topic}")

print(f"\n⏱️  Estimated study time: {plan['estimated_time_minutes']} minutes")

### Advanced State
When most topics are mastered, the engine surfaces the most advanced concepts.

In [ ]:
# Master remaining topics
tracker.record_score("Data Structures", 90)
tracker.record_score("Data Structures", 92)
tracker.record_score("Algorithms", 88)
tracker.record_score("Algorithms", 91)

recs = engine.get_study_recommendations()
print("=== Advanced Recommendations ===")
for r in recs:
    print(f"  [{r['priority']}] {r['action'].upper()}: {r['topic']} — {r['reason']}")

print(f"\nNext topics: {engine.suggest_next_topics()}")

In [ ]:
# Cleanup demo files
import os
for f in ["./demo_progress.json", "./demo_sr.json"]:
    if os.path.exists(f):
        os.remove(f)
print("Demo files cleaned up.")